# tensor-zeros-init — worked example 3: Scatter event counts into a zeros buffer

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `tensor-zeros-init`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Concept

The 'allocate-then-fill' idiom: allocate a `(n_bins,)` integer zero buffer with `dtype=t.long`, then scatter-accumulate per-element results into it with `index_add_`. Integer dtype is required because counts are whole numbers and the buffer is later usable as a histogram. This beats append-and-stack for fixed-size outputs.

## Worked solution

We have a list of integer bin ids and want to count how many landed in each of `n_bins` bins.

1. We allocate the counter buffer with `t.zeros(n_bins, dtype=t.long)`. The explicit `dtype=t.long` keeps the counts as exact int64 values.
2. `counts.index_add_(0, bins, t.ones_like(bins))` adds 1 to `counts[b]` for every `b` in `bins`, in a single vectorized scatter along axis 0.
3. Because every sample contributes exactly one increment, the buffer's total must equal the number of samples — we print that as a sanity check.

The printed per-bin counts show the distribution we scattered.

In [ ]:
import torch as t

def histogram(bins: t.Tensor, n_bins: int) -> t.Tensor:
    counts = t.zeros(n_bins, dtype=t.long)
    counts.index_add_(0, bins, t.ones_like(bins))
    return counts

bins = t.tensor([0, 2, 2, 1, 4, 2, 0], dtype=t.long)
counts = histogram(bins, 5)
print('counts:', counts.tolist())
print('total matches:', counts.sum().item() == bins.numel())